# Proximity Mapping Visualization
## Trentino Weather Stations & EEA Air Quality Stations

This notebook visualizes the proximity relationships between Trentino weather monitoring stations and nearby EEA air quality stations using interactive maps and statistical analysis.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Load Proximity Mapping Data

In [2]:
# Load the proximity mapping data
proximity_mapping = pd.read_csv('../output/trentino_eea_proximity_mapping.csv')

print("Proximity Mapping Data Shape:", proximity_mapping.shape)
print("\nFirst 10 rows:")
print(proximity_mapping.head(10))
print("\nData Summary:")
print(f"Unique Trentino Stations: {proximity_mapping['Trentino_Stazione'].nunique()}")
print(f"Unique EEA Stations: {proximity_mapping['EEA_Stazione'].nunique()}")
print(f"Total Station Pairs: {len(proximity_mapping)}")

Proximity Mapping Data Shape: (80, 2)

First 10 rows:
  Trentino_Stazione                             EEA_Stazione
0        A22 (Avio)   SPO.IT1848A_5_BETA_2007-01-15_00:00:00
1        A22 (Avio)  SPO.IT0663A_5_gravi_2003-10-15_00:00:00
2        A22 (Avio)   SPO.IT1343A_5_BETA_2004-01-14_00:00:00
3        A22 (Avio)   SPO.IT1336A_5_BETA_2003-10-15_00:00:00
4        A22 (Avio)   SPO.IT1388A_5_BETA_2006-12-19_00:00:00
5        A22 (Avio)   SPO.IT1340A_5_BETA_2009-08-01_00:00:00
6        A22 (Avio)   SPO.IT0742A_5_BETA_2000-06-01_00:00:00
7        A22 (Avio)   SPO.IT1177A_5_BETA_2002-02-01_00:00:00
8        A22 (Avio)  SPO.IT1838A_5_gravi_2007-01-01_00:00:00
9        A22 (Avio)   SPO.IT0846A_5_BETA_2002-11-18_00:00:00

Data Summary:
Unique Trentino Stations: 8
Unique EEA Stations: 80
Total Station Pairs: 80


## 3. Load Station Coordinates

In [3]:
# Load Trentino stations with coordinates
trentino_data = pd.read_csv('../output/historical_weather_airPM_trentino.csv')
trentino_stations = trentino_data[['Stazione', 'Latitudine', 'Longitudine']].drop_duplicates()

# Load filtered EEA stations with coordinates
eea_data = pd.read_csv('../output/eea_filtered_by_proximity.csv')
eea_stations = eea_data[['Stazione', 'Latitudine', 'Longitudine']].drop_duplicates()

print("Trentino Stations:")
print(trentino_stations)
print(f"\nTotal: {len(trentino_stations)} unique stations")
print(f"\nEEA Stations (filtered):")
print(f"Total: {len(eea_stations)} unique stations")
print(f"\nEEA Data Shape: {eea_data.shape}")
print(f"Date Range: {eea_data['Data'].min()} to {eea_data['Data'].max()}")

Trentino Stations:
              Stazione  Latitudine  Longitudine
0           A22 (Avio)    45.74215     10.97043
3965   Borgo Valsugana    46.05184     11.45389
8308        Monte Gaza    46.08253     10.95804
12599  Parco S. Chiara    46.06292     11.12620
16976  Piana Rotaliana    46.19683     11.11343
21248   Riva del Garda    45.89146     10.84448
25597         Rovereto    45.89243     11.03941
29935      Via Bolzano    46.10433     11.11022

Total: 8 unique stations

EEA Stations (filtered):
Total: 80 unique stations

EEA Data Shape: (260232, 9)
Date Range: 2013-01-01 to 2024-12-31


## 4. Create Interactive Map Visualization

In [4]:
# Create interactive map
center_lat = trentino_stations['Latitudine'].mean()
center_lon = trentino_stations['Longitudine'].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles='OpenStreetMap'
)

# Add Trentino stations (blue markers)
for idx, row in trentino_stations.iterrows():
    folium.CircleMarker(
        location=[row['Latitudine'], row['Longitudine']],
        radius=10,
        popup=f"<b>Trentino:</b> {row['Stazione']}",
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.8,
        weight=2
    ).add_to(m)

# Add EEA stations (red markers)
for idx, row in eea_stations.iterrows():
    folium.CircleMarker(
        location=[row['Latitudine'], row['Longitudine']],
        radius=6,
        popup=f"<b>EEA:</b> {row['Stazione']}",
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.6,
        weight=1
    ).add_to(m)

# Draw lines connecting Trentino stations to their assigned EEA stations
for idx, row in proximity_mapping.iterrows():
    trentino_name = row['Trentino_Stazione']
    eea_name = row['EEA_Stazione']
    
    # Get coordinates
    trentino_coords = trentino_stations[trentino_stations['Stazione'] == trentino_name]
    eea_coords = eea_stations[eea_stations['Stazione'] == eea_name]
    
    if len(trentino_coords) > 0 and len(eea_coords) > 0:
        t_lat, t_lon = trentino_coords.iloc[0][['Latitudine', 'Longitudine']]
        e_lat, e_lon = eea_coords.iloc[0][['Latitudine', 'Longitudine']]
        
        folium.PolyLine(
            locations=[[t_lat, t_lon], [e_lat, e_lon]],
            color='green',
            weight=1,
            opacity=0.4
        ).add_to(m)

# Add legend
legend_html = '''
     <div style="position: fixed; 
                     bottom: 50px; left: 50px; width: 220px; height: 140px; 
                     background-color: white; border:2px solid grey; z-index:9999; 
                     font-size:14px; padding: 10px">
     <p style="margin:0; font-weight: bold;">Legend</p>
     <p style="margin:5px 0;"><i style="background:blue; width:15px; height:15px; 
        display:inline-block; border-radius:50%;"></i> Trentino Weather Stations</p>
     <p style="margin:5px 0;"><i style="background:red; width:12px; height:12px; 
        display:inline-block; border-radius:50%;"></i> EEA Air Quality Stations</p>
     <p style="margin:5px 0;"><i style="background:green; width:20px; height:2px; 
        display:inline-block;"></i> Station Proximity Links</p>
     </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

m

## 8. Verify EEA Coordinates from Filtered CSV

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

In [6]:
eea_filtered_csv = pd.read_csv('../output/eea_filtered_by_proximity.csv')

In [7]:
# Extract unique station coordinates from the filtered CSV
unique_stations_csv = eea_filtered_csv[['Stazione', 'Latitudine', 'Longitudine']].drop_duplicates().reset_index(drop=True)

print(f"Total unique EEA stations in filtered CSV: {len(unique_stations_csv)}")
print("\nUnique Stations:")
print(unique_stations_csv.to_string(index=False))

# Create interactive map with unique stations
center_lat = unique_stations_csv['Latitudine'].mean()
center_lon = unique_stations_csv['Longitudine'].mean()

m_filtered = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles='OpenStreetMap'
)

# Add each unique station as a marker
for idx, row in unique_stations_csv.iterrows():
    folium.CircleMarker(
        location=[row['Latitudine'], row['Longitudine']],
        radius=8,
        popup=f"<b>Station:</b> {row['Stazione']}<br><b>Lat:</b> {row['Latitudine']:.5f}<br><b>Lon:</b> {row['Longitudine']:.5f}",
        color='purple',
        fill=True,
        fillColor='purple',
        fillOpacity=0.7,
        weight=2
    ).add_to(m_filtered)

# Add legend
legend_html = '''
     <div style="position: fixed; 
                     bottom: 50px; left: 50px; width: 220px; height: 100px; 
                     background-color: white; border:2px solid grey; z-index:9999; 
                     font-size:14px; padding: 10px">
     <p style="margin:0; font-weight: bold;">EEA Filtered Stations</p>
     <p style="margin:5px 0;"><i style="background:purple; width:15px; height:15px; 
        display:inline-block; border-radius:50%;"></i> Unique EEA Stations</p>
     <p style="margin:5px 0; font-size:12px;">Total: {}</p>
     </div>
     '''.format(len(unique_stations_csv))
m_filtered.get_root().html.add_child(folium.Element(legend_html))

m_filtered

Total unique EEA stations in filtered CSV: 80

Unique Stations:
                               Stazione  Latitudine  Longitudine
 SPO.IT1619A_5_BETA_2004-05-04_00:00:00    46.03083     11.90583
 SPO.IT0499A_5_BETA_2004-10-09_00:00:00    46.61947     10.85906
 SPO.IT1918A_5_BETA_2008-11-20_00:00:00    44.82500     11.64972
 SPO.IT1097A_5_BETA_2022-11-07_00:00:00    45.89865     12.53631
 SPO.IT0908A_5_BETA_2003-04-10_00:00:00    46.47028     10.37250
 SPO.IT1910A_5_BETA_2008-03-05_00:00:00    44.92611     10.37083
 SPO.IT2201A_5_BETA_2014-07-18_00:00:00    45.96262     12.65594
 SPO.IT1871A_5_BETA_2008-01-08_00:00:00    45.22667     11.66583
 SPO.IT2063A_5_BETA_2011-02-16_00:00:00    45.14833      9.94028
 SPO.IT0839A_5_BETA_2007-09-25_00:00:00    45.36722      9.70500
 SPO.IT0906A_5_BETA_1999-10-01_00:00:00    46.16722      9.87056
 SPO.IT2300A_5_BETA_2021-01-01_00:00:00    45.53861     10.23194
 SPO.IT0777A_5_BETA_1999-11-01_00:00:00    45.69796      9.40619
 SPO.IT1862A_5_BETA_2011-1